<a href="https://colab.research.google.com/github/Srinidhi08092006/codesoft/blob/main/project_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

print("ROOT:", os.listdir("."))
print("MODULES EXISTS:", os.path.exists("modules"))
if os.path.exists("modules"):
    print("MODULES CONTENT:", os.listdir("modules"))


ROOT: ['data', 'drive', '.ipynb_checkpoints', 'modules', 'outputs', 'sample_data']
MODULES EXISTS: True
MODULES CONTENT: ['__init__.py', '__pycache__', 'recommender.py', 'preprocessing.py', 'validator.py', 'llm_ensemble.py', '.ipynb_checkpoints']


In [2]:
import os

folders = [
    "modules",
    "data/sample_inputs",
    "outputs"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("✅ Project structure created")


✅ Project structure created


In [3]:
# modules/preprocessing.py
import json
import numpy as np

def preprocess_inputs(labs, wearables, questionnaire):
    features = {}
    qc_flags = []
    missing_flags = []

    # ---- Labs ----
    for biomarker, value in labs.items():
        if value is None:
            missing_flags.append(biomarker)
        else:
            features[biomarker] = float(value)

            if value < 0:
                qc_flags.append(f"{biomarker}_NEGATIVE")

    # ---- Wearables ----
    for k, v in wearables.items():
        if v is None:
            missing_flags.append(k)
        else:
            features[k] = v

    # ---- Questionnaire ----
    for k, v in questionnaire.items():
        if v is None:
            missing_flags.append(k)
        else:
            features[k] = v

    return features, qc_flags, missing_flags


def save_preprocessing_outputs(features, qc, missing):
    with open("outputs/features.json", "w") as f:
        json.dump(features, f, indent=2)

    with open("outputs/qc_flags.json", "w") as f:
        json.dump(qc, f, indent=2)

    with open("outputs/missing_flags.json", "w") as f:
        json.dump(missing, f, indent=2)


In [4]:
# modules/llm_ensemble.py
import random

MODEL_NAMES = [
    "Command-R",
    "Qwen",
    "LLaMA",
    "Mistral",
    "Phi",
    "Falcon",
    "Gemma"
]

def run_llm(model_name, features):
    # deterministic mock output
    return {
        "model": model_name,
        "domain_scores": {
            "neuro": round(random.uniform(0.6, 0.9), 2),
            "metabolic": round(random.uniform(0.5, 0.8), 2)
        },
        "ranked_focus_areas": ["neuro", "metabolic"],
        "suggested_actions": ["SUPPORT_NEURO", "CONTROL_GLUCOSE"],
        "rationale_map": {
            "neuro": "Sleep and glucose patterns indicate risk"
        }
    }

def run_ensemble(features):
    outputs = []
    for model in MODEL_NAMES:
        outputs.append(run_llm(model, features))
    return outputs


In [5]:
# modules/validator.py
ACTION_WHITELIST = {"SUPPORT_NEURO", "CONTROL_GLUCOSE"}

def validate_outputs(model_outputs):
    valid = []
    log = []

    for out in model_outputs:
        actions = set(out["suggested_actions"])
        if not actions.issubset(ACTION_WHITELIST):
            log.append(f"{out['model']}: INVALID ACTION")
            continue
        valid.append(out)

    return valid, log
def ensemble_decision(valid_outputs):
    neuro_scores = [o["domain_scores"]["neuro"] for o in valid_outputs]
    avg_neuro = round(sum(neuro_scores) / len(neuro_scores), 2)

    final_focus = "neuro" if avg_neuro >= 0.7 else "metabolic"

    return {
        "final_focus": final_focus,
        "avg_neuro_score": avg_neuro
    }
# modules/recommender.py
def recommend(decision):
    recs = []

    if decision["final_focus"] == "neuro":
        recs.append({
            "nutraceutical": "Omega-3",
            "dose_range": "1000–2000 mg/day",
            "rationale": "Supports neurovascular function",
            "review_required": False
        })

    return recs


In [6]:
# Create __init__.py so Python treats modules/ as a package
open("modules/__init__.py", "w").close()

print("✅ modules package initialized")
import sys
import os

sys.path.append(os.getcwd())

print("✅ Project root added to sys.path")


✅ modules package initialized
✅ Project root added to sys.path


In [7]:
import os

print("CWD:", os.getcwd())
print("ROOT FILES:", os.listdir("/content"))
print("MODULE FILES:", os.listdir("/content/modules"))


CWD: /content
ROOT FILES: ['data', 'drive', '.ipynb_checkpoints', 'modules', 'outputs', 'sample_data']
MODULE FILES: ['__init__.py', '__pycache__', 'recommender.py', 'preprocessing.py', 'validator.py', 'llm_ensemble.py', '.ipynb_checkpoints']


In [8]:
from modules.preprocessing import preprocess_inputs, save_preprocessing_outputs
from modules.llm_ensemble import run_ensemble
from modules.validator import validate_outputs, ensemble_decision
from modules.recommender import recommend
import json



# ---- SAMPLE INPUTS ----
labs = {
    "Glucose": 132,
    "VitaminD": 18
}

wearables = {
    "avg_heart_rate": 88,
    "sleep_hours": 5.8
}

questionnaire = {
    "stress_level": 4,
    "exercise": "low"
}

# ---- PIPELINE ----
features, qc, missing = preprocess_inputs(labs, wearables, questionnaire)
save_preprocessing_outputs(features, qc, missing)

llm_outputs = run_ensemble(features)
valid_outputs, validation_log = validate_outputs(llm_outputs)

decision = ensemble_decision(valid_outputs)
recs = recommend(decision)

# ---- SAVE OUTPUTS ----
with open("outputs/ensemble_decision.json", "w") as f:
    json.dump(decision, f, indent=2)

with open("outputs/nutraceutical_recommendations.json", "w") as f:
    json.dump(recs, f, indent=2)

with open("outputs/validation_log.json", "w") as f:
    json.dump(validation_log, f, indent=2)

print("✅ TASK-4 PIPELINE EXECUTED SUCCESSFULLY")


✅ TASK-4 PIPELINE EXECUTED SUCCESSFULLY


In [9]:
import sys, os
sys.path.append(os.getcwd())

from modules.preprocessing import preprocess_inputs, save_preprocessing_outputs
from modules.llm_ensemble import run_ensemble
from modules.validator import validate_outputs, ensemble_decision
from modules.recommender import recommend

print("✅ IMPORTS FIXED. ALL MODULES LOADED.")


✅ IMPORTS FIXED. ALL MODULES LOADED.
